# Phân tích baseline v6
Planner quyết định từng bước bằng instruction ngắn; caller chọn tool và arguments. Sau mỗi kết quả thành công, quay lại planner; lỗi thì dừng. Không có tập tool khóa từ đầu như v5.


In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
from IPython.display import display, Markdown

candidates = [Path("/kaggle/working/lifemate_baseline_v6"), Path("baseline_v6/outputs"), Path("outputs"), Path("../outputs")]
ROOT = next((p for p in candidates if p.exists()), candidates[0])  # Đổi ROOT nếu report nằm chỗ khác

def jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()] if path.exists() else []

def read_run(mode):
    base = ROOT / mode
    return {
        "summary": json.loads((base/"summary.json").read_text()) if (base/"summary.json").exists() else {},
        "cases": pd.DataFrame(jsonl(base/"traces.jsonl")),
        "metrics": pd.read_csv(base/"cases.csv") if (base/"cases.csv").exists() and (base/"cases.csv").stat().st_size else pd.DataFrame(),
        "status": json.loads((base/"run_status.json").read_text()) if (base/"run_status.json").exists() else {},
    }
runs = {mode: read_run(mode) for mode in ("workflow_gold", "baseline_asr")}
for mode, run in runs.items():
    print(mode, run["status"])
    display(run["summary"])


## Metric và lỗi
`invalid_model_output`: sai contract. `validation_failed`/`tool_failed`: lỗi schema/nghiệp vụ, dừng không retry. `give_up`: planner nhận định không thể hoàn thành. `done`: planner nhận định đã xong, cần đối chiếu label. `environment_no_progress`/`duplicate_write`: guard code.


In [ ]:
for mode, run in runs.items():
    df = run["metrics"]
    if df.empty:
        continue
    print(mode)
    display(df[[c for c in ["id", "score_proxy", "state_hash_match", "stop_reason", "tool_count", "planner_calls", "caller_calls", "llm_calls", "agent_latency_s"] if c in df]])
    display(df.stop_reason.value_counts().rename("count").to_frame())
    display(df.loc[df.score_proxy < 1, ["id", "error_flags", "missing_tool_names", "extra_tool_names"]])


## Chi phí hai node
Mỗi decision là một lượt planner, kèm tối đa một caller. Trace node_outputs có plan/call; lượt done/give_up chỉ có plan. So số lượt LLM bằng llm_calls, không dùng decisions để so trực tiếp với v5. Batch giữ prompt/schema và session riêng cho từng mẫu.


In [ ]:
node_rows = []
for mode, run in runs.items():
    for case in run["cases"].to_dict("records"):
        for event in case.get("trace", []):
            for node, output in event.get("node_outputs", {}).items():
                node_rows.append({"mode": mode, "id": case["id"], "decision": event["decision"], "node": node,
                    **{k: output.get(k) for k in ["prompt_tokens", "generation_tokens", "latency_s"]}})
nodes = pd.DataFrame(node_rows)
if not nodes.empty:
    display(nodes.groupby(["mode", "node"])[["prompt_tokens", "generation_tokens", "latency_s"]].agg(["count", "mean", "max"]))


In [ ]:
def show_case(sample_id, mode="workflow_gold"):
    run = runs[mode]
    traces = run["cases"]
    if traces.empty or sample_id not in set(traces.id):
        print("Không tìm thấy", sample_id)
        return
    metric = run["metrics"]
    if not metric.empty:
        display(metric.loc[metric.id == sample_id].T)
    case = traces.loc[traces.id == sample_id].iloc[0].to_dict()
    print(json.dumps(case, ensure_ascii=False, indent=2))

# show_case("<sample_id>")


## Diễn giải
- Kiểm planner_output.instruction: đúng bước chưa hoàn thành hay đang bỏ sót điều kiện/hành động?
- Kiểm caller action: đúng tool, field, value và đúng dữ kiện đã quan sát?
- Instruction là guidance của model, không phải bằng chứng ID/giá trị đã tồn tại.
- Env.validate chỉ kiểm schema; ok=true/done=false là thất bại nghiệp vụ.
- Không có source/grounding reviewer. Local diagnostic là log code, không gửi lại model làm feedback.
- Dùng wall time/throughput cho batch và workers=1 cho latency riêng. Chưa có số liệu GPU chứng minh cải thiện.
